# Forecasting Space Radiation with LSTMs

**Solar Proton Events** (SPEs) are bursts of high-energy particles ejected by the Sun that can deliver dangerous radiation doses to astronauts and spacecraft within hours. Accurate forecasting of these events is critical for future crewed missions beyond low-Earth orbit.

This notebook demonstrates a deep learning approach to SPE forecasting developed by the [FDL-X 2024 Radiation Team](https://frontierdevelopmentlab.org/). The model, **RadRecurrentWithSDO**, is a dual-LSTM architecture that:

1. **Observes** a 10-hour context window of solar imagery (SDO/AIA + HMI, 6 channels at 512x512) and time-series data (GOES X-ray flux, BioSentinel dose rate).
2. **Encodes** each SDO image through a CNN into a compact embedding, which is concatenated with the time-series and fed through a context LSTM to build up a representation of the current solar state.
3. **Forecasts** future radiation levels by transferring the context LSTM's hidden state to a prediction LSTM that autoregressively generates the next 10 hours of output at 15-minute resolution.

Uncertainty is estimated via **Monte Carlo dropout** -- running inference 50 times with dropout active to produce a distribution of predictions from a single model.

We'll run it on **biosentinel07**, a major SPE from July 18, 2023 that reached 620 pfu (particle flux units) at >10 MeV -- one of the strongest events in the BioSentinel mission.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FrontierDevelopmentLab/2024-hl-radiation-ml/blob/main/public/demo.ipynb)

## Install dependencies and clone the repo

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q duckdb sunpy

# Clone the repo to get model definitions, dataset classes, and event catalog
!git clone -q https://github.com/FrontierDevelopmentLab/2024-hl-radiation-ml.git repo

In [ ]:
import sys, os, datetime
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates

sys.path.insert(0, 'repo/scripts')

from datasets import SDOMLlite, RadLab, GOESXRS, Sequences
from models import RadRecurrentWithSDO
from events import EventCatalog

EVENT_ID = 'biosentinel07'   # July 2023, 620 pfu
NUM_SAMPLES = 50             # MC-dropout samples
DELTA_MINUTES = 15           # time step
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Download data from S3

All data is hosted on a public S3 bucket. We download:
- **Model checkpoint** (~1.4 GB) -- trained `RadRecurrentWithSDO` weights
- **GOES XRS** (~157 MB) -- X-ray flux CSV
- **RadLab DuckDB** (~311 MB) -- BioSentinel BPD dose rate database
- **SDOML-lite tar files** (4 files, ~3 GB) -- SDO imagery covering the event window

In [ ]:
S3_BASE = 'https://nasa-radiant-data.s3.amazonaws.com/helioai-datasets/hl-dosi'
DATA_DIR = 'data'

# Create local directory structure
os.makedirs(f'{DATA_DIR}/goes', exist_ok=True)
os.makedirs(f'{DATA_DIR}/radlab', exist_ok=True)
os.makedirs(f'{DATA_DIR}/sdoml-lite-biosentinel', exist_ok=True)
os.makedirs(f'{DATA_DIR}/models', exist_ok=True)

# Download files (wget -nc skips if already present)
!wget -q -nc -P {DATA_DIR}/models      {S3_BASE}/models/epoch-002-model-20240905.pth
!wget -q -nc -P {DATA_DIR}/goes        {S3_BASE}/goes/goes-xrs.csv
!wget -q -nc -P {DATA_DIR}/radlab      {S3_BASE}/radlab/RadLab-20240625-duck.db
!wget -q -nc -P {DATA_DIR}/sdoml-lite-biosentinel {S3_BASE}/sdoml-lite-biosentinel/sdoml-lite-biosentinel-258.tar
!wget -q -nc -P {DATA_DIR}/sdoml-lite-biosentinel {S3_BASE}/sdoml-lite-biosentinel/sdoml-lite-biosentinel-259.tar
!wget -q -nc -P {DATA_DIR}/sdoml-lite-biosentinel {S3_BASE}/sdoml-lite-biosentinel/sdoml-lite-biosentinel-260.tar
!wget -q -nc -P {DATA_DIR}/sdoml-lite-biosentinel {S3_BASE}/sdoml-lite-biosentinel/sdoml-lite-biosentinel-261.tar

!du -sh {DATA_DIR}/*

## Load the model

The checkpoint stores both the architecture hyperparameters and the trained weights, so we can reconstruct the model directly from the file.

In [ ]:
MODEL_FILE = f'{DATA_DIR}/models/epoch-002-model-20240905.pth'
checkpoint = torch.load(MODEL_FILE, map_location=DEVICE, weights_only=False)

model = RadRecurrentWithSDO(
    data_dim=checkpoint['model_data_dim'],
    lstm_dim=checkpoint['model_lstm_dim'],
    lstm_depth=checkpoint['model_lstm_depth'],
    dropout=checkpoint['model_dropout'],
    context_window=checkpoint['model_context_window'],
    prediction_window=checkpoint['model_prediction_window'],
    sdo_channels=checkpoint['model_sdo_channels'],
    sdo_dim=checkpoint['model_sdo_dim'],
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f'Parameters       : {num_params:,}')
print(f'Data dimensions  : {model.data_dim}')
print(f'Context window   : {model.context_window} steps ({model.context_window * DELTA_MINUTES / 60:.0f} hours)')
print(f'Prediction window: {model.prediction_window} steps ({model.prediction_window * DELTA_MINUTES / 60:.0f} hours)')
print(f'Trained epochs   : {checkpoint["epoch"] + 1}')

## Select a Solar Proton Event and load datasets

The event catalog contains SPEs observed by BioSentinel and CRaTER. We load three data sources covering the event window:
- **SDOML-lite**: SDO solar imagery stored as numpy arrays inside tar files (15-min cadence)
- **GOES XRS**: X-ray flux from GOES-16 (1-min cadence, log-normalized)
- **BioSentinel BPD**: Absorbed dose rate in deep space (1-min cadence, log-normalized)

The `Sequences` class finds contiguous windows where all datasets have data available.

In [ ]:
date_start_str, date_end_str, max_pfu = EventCatalog[EVENT_ID]
date_start = datetime.datetime.fromisoformat(date_start_str)
date_end = datetime.datetime.fromisoformat(date_end_str)
context_start = date_start - datetime.timedelta(minutes=(model.context_window - 1) * DELTA_MINUTES)

print(f'Event        : {EVENT_ID}')
print(f'>10 MeV peak : {max_pfu} pfu')
print(f'Event window : {date_start} to {date_end}')
print(f'Context from : {context_start}')

dataset_sdo = SDOMLlite(
    f'{DATA_DIR}/sdoml-lite-biosentinel',
    date_start=context_start, date_end=date_end
)
dataset_goes_xrs = GOESXRS(
    f'{DATA_DIR}/goes/goes-xrs.csv',
    date_start=context_start, date_end=date_end
)
dataset_biosentinel = RadLab(
    f'{DATA_DIR}/radlab/RadLab-20240625-duck.db',
    instrument='BPD', date_start=context_start, date_end=date_end
)

In [ ]:
# Find contiguous sequences where all three datasets have data
sequences = Sequences(
    [dataset_sdo, dataset_goes_xrs, dataset_biosentinel],
    delta_minutes=DELTA_MINUTES,
    sequence_length=model.context_window
)
print(f'\n{len(sequences)} valid context sequences found')

## Prepare context window

Extract the first valid sequence as the model's context input. The model architecture has two phases:

1. **Context phase**: A context LSTM ingests SDO image embeddings concatenated with time-series data, building up hidden state.
2. **Prediction phase**: The hidden state is transferred to a prediction LSTM, which autoregressively generates future time steps.

In [ ]:
# Unpack the first sequence: (sdo_images, xrs_values, bpd_values, date_strings)
context_sequence = sequences[0]
context_dates = [datetime.datetime.fromisoformat(d) for d in context_sequence[3]]
context_end = context_dates[-1]
prediction_window = int((date_end - context_end).total_seconds() / (DELTA_MINUTES * 60))

# Prepare tensors
context_sdo = context_sequence[0][:model.context_window].to(DEVICE).unsqueeze(0)        # (1, T, C, H, W)
ctx_xrs = context_sequence[1][:model.context_window].unsqueeze(1).to(DEVICE)             # (T, 1)
ctx_bio = context_sequence[2][:model.context_window].unsqueeze(1).to(DEVICE)             # (T, 1)
context_data = torch.cat([ctx_xrs, ctx_bio], dim=1).unsqueeze(0)                         # (1, T, 2)

print(f'Context    : {context_dates[0]} to {context_end}')
print(f'Predicting : {prediction_window} steps ({prediction_window * DELTA_MINUTES / 60:.0f} hours)')
print(f'SDO shape  : {context_sdo.shape}')
print(f'Data shape : {context_data.shape}')

## Run inference with Monte Carlo dropout

We keep the model in **train mode** so that dropout remains active. Each forward pass produces a different prediction due to the random dropout mask. Running `NUM_SAMPLES` passes gives us a distribution we can summarize with mean and spread.

In [ ]:
model.train()  # keep dropout active for MC sampling

with torch.no_grad():
    predictions = model.predict(
        context_sdo, context_data, prediction_window, num_samples=NUM_SAMPLES
    ).detach()

print(f'Prediction tensor: {predictions.shape}  (samples, time steps, channels)')

## Unnormalize and prepare for plotting

The model operates on log-normalized data. We convert both the predictions and ground truth back to physical units for plotting.

In [ ]:
prediction_dates = [
    context_end + datetime.timedelta(minutes=i * DELTA_MINUTES)
    for i in range(prediction_window + 1)
]
training_pred_end = context_end + datetime.timedelta(
    minutes=model.prediction_window * DELTA_MINUTES
)

# Unnormalize predictions (channel 0 = XRS, channel 1 = BPD)
xrs_preds = dataset_goes_xrs.unnormalize_data(predictions[:, :, 0]).cpu().numpy()
bio_preds = dataset_biosentinel.unnormalize_data(predictions[:, :, 1]).cpu().numpy()

# Ground truth over the full window
context_start = context_dates[0]
xrs_gt_dates, xrs_gt_vals = dataset_goes_xrs.get_series(context_start, date_end, delta_minutes=DELTA_MINUTES)
bio_gt_dates, bio_gt_vals = dataset_biosentinel.get_series(context_start, date_end, delta_minutes=DELTA_MINUTES)
xrs_gt_vals = dataset_goes_xrs.unnormalize_data(xrs_gt_vals)
bio_gt_vals = dataset_biosentinel.unnormalize_data(bio_gt_vals)

## Plot predictions vs ground truth

The plot shows:
- **Blue/purple**: observed ground truth
- **Light red traces**: individual MC-dropout prediction samples
- **Red line**: mean prediction across all samples
- **Vertical lines**: context start (dashed), prediction start (solid), training prediction horizon (dashed)

In [ ]:
fig, (ax_bio, ax_xrs) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
hours_loc = matplotlib.dates.HourLocator(interval=2)
date_fmt = matplotlib.dates.DateFormatter('%m-%d %H:%M')

for ax, gt_dates, gt_vals, preds, color, title, ylabel in [
    (ax_bio, bio_gt_dates, bio_gt_vals, bio_preds, 'mediumblue',
     'BioSentinel BPD', 'Absorbed dose rate [\u03bcGy/min]'),
    (ax_xrs, xrs_gt_dates, xrs_gt_vals, xrs_preds, 'purple',
     'GOES XRS', 'X-ray flux [W/m\u00b2]'),
]:
    # Ground truth
    ax.plot(gt_dates, gt_vals, color=color, alpha=0.75, label='Ground truth')

    # MC-dropout samples
    for s in range(NUM_SAMPLES):
        ax.plot(prediction_dates, preds[s], color='red', alpha=0.05,
                label='Prediction (samples)' if s == 0 else None)

    # Prediction mean
    ax.plot(prediction_dates, np.mean(preds, axis=0), color='red', alpha=0.7,
            label='Prediction (mean)')

    # Boundary markers
    ax.axvline(context_dates[0], color='red', ls='--', lw=1)
    ax.axvline(prediction_dates[0], color='red', ls='-', lw=1.5)
    ax.axvline(training_pred_end, color='red', ls='--', lw=1)

    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_yscale('log')
    ax.xaxis.set_minor_locator(hours_loc)
    ax.grid(color='#f0f0f0', which='minor', axis='x')
    ax.grid(color='lightgray', which='major')
    ax.legend(loc='upper right')

ax_xrs.xaxis.set_major_formatter(date_fmt)
fig.suptitle(f'Event: {EVENT_ID} (>10 MeV max: {max_pfu} pfu)  |  '
             f'Prediction from {prediction_dates[0]:%Y-%m-%d %H:%M}')
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()